# Load NHTSA complaints into Pinecone

Seeds the `carverdict` index with real complaint data for a wide set of popular US vehicles, so the chat works out of the box for the makes people actually ask about. Run this once to (re)load the index.

In [1]:
import os, json, time, warnings
import requests
from dotenv import load_dotenv
from pinecone import Pinecone

warnings.filterwarnings("ignore")
load_dotenv(".env.local")

GEMINI_KEY = os.environ["GEMINI_API_KEY"]
PINECONE_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME = os.environ["PINECONE_INDEX"]

/Users/sufiyanrauf/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Popular US vehicles to seed. NHTSA's complaints API has its own model vocabulary,
# so some entries query a different string than the clean name we store (e.g. the F-150
# is filed under "F-150 SUPERCAB", Lexus RX 350 under "RX350", Mercedes by class).
vehicles = [
    ("Toyota","Camry",2019,"Camry"), ("Toyota","Corolla",2019,"Corolla"),
    ("Toyota","RAV4",2019,"RAV4"), ("Toyota","Highlander",2020,"Highlander"),
    ("Toyota","Tacoma",2019,"Tacoma"),
    ("Honda","Accord",2019,"Accord"), ("Honda","Civic",2019,"Civic"),
    ("Honda","CR-V",2019,"CR-V"), ("Honda","Pilot",2019,"Pilot"),
    ("Honda","Odyssey",2019,"Odyssey"),
    ("Ford","Explorer",2020,"Explorer"), ("Ford","Escape",2020,"Escape"),
    ("Ford","Mustang",2019,"Mustang"), ("Ford","F-150",2020,"F-150 SUPERCAB"),
    ("Chevrolet","Silverado 1500",2020,"Silverado 1500"), ("Chevrolet","Equinox",2019,"Equinox"),
    ("Chevrolet","Malibu",2019,"Malibu"), ("Chevrolet","Corvette",2019,"Corvette"),
    ("Tesla","Model 3",2021,"Model 3"), ("Tesla","Model Y",2020,"Model Y"),
    ("Nissan","Altima",2019,"Altima"), ("Nissan","Rogue",2019,"Rogue"),
    ("Nissan","Sentra",2019,"Sentra"),
    ("Jeep","Grand Cherokee",2019,"Grand Cherokee"), ("Jeep","Wrangler",2019,"Wrangler"),
    ("Hyundai","Elantra",2019,"Elantra"), ("Hyundai","Tucson",2019,"Tucson"),
    ("Subaru","Outback",2019,"Outback"),
    ("Ram","1500",2019,"1500"), ("GMC","Sierra 1500",2019,"Sierra 1500"),
    ("Lexus","RX 350",2019,"RX350"), ("Lexus","ES 350",2019,"ES 350"),
    ("Mercedes-Benz","E-Class",2019,"E-CLASS"), ("Mercedes-Benz","C-Class",2019,"C-CLASS"),
    ("BMW","3 Series",2019,"3 SERIES"), ("BMW","X5",2019,"X5"),
    ("Audi","Q5",2019,"Q5"),
    ("Cadillac","Escalade",2019,"Escalade"), ("Cadillac","XT5",2019,"XT5"),
    ("Dodge","Charger",2019,"Charger"), ("Dodge","Durango",2019,"Durango"),
    ("Dodge","Challenger",2019,"Challenger"),
    ("Volkswagen","Jetta",2019,"Jetta"), ("Volkswagen","Tiguan",2019,"Tiguan"),
    ("Volkswagen","Atlas",2019,"Atlas"),
    ("Kia","Telluride",2020,"Telluride"), ("Kia","Sorento",2019,"Sorento"),
    ("Kia","Forte",2019,"Forte"),
    ("Mazda","CX-5",2019,"CX-5"), ("Mazda","Mazda3",2019,"Mazda3"),
    ("Acura","MDX",2019,"MDX"), ("Chrysler","Pacifica",2019,"Pacifica"),
    ("Infiniti","QX60",2019,"QX60"), ("Buick","Encore",2019,"Encore"),
]
per_vehicle = 4

def get_complaints(make, query_model, year):
    url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
    r = requests.get(url, params={"make": make, "model": query_model, "modelYear": year}, timeout=30)
    r.raise_for_status()
    return r.json().get("results", [])

records = []
for make, model, year, query_model in vehicles:
    results = get_complaints(make, query_model, year)
    print(f"{make} {model} {year}: {len(results)} complaints found")
    for c in results[:per_vehicle]:
        summary = (c.get("summary") or "").strip()
        if not summary:
            continue
        records.append({
            "id": str(c["odiNumber"]), "make": make, "model": model, "year": year,
            "components": [x.strip() for x in (c.get("components") or "").split(",") if x.strip()],
            "date": c.get("dateComplaintFiled", ""),
            "crash": bool(c.get("crash")), "fire": bool(c.get("fire")),
            "injuries": c.get("numberOfInjuries", 0), "deaths": c.get("numberOfDeaths", 0),
            "summary": summary,
        })

print("total records:", len(records))

Toyota Camry 2019: 379 complaints found
Toyota Corolla 2019: 200 complaints found


Toyota RAV4 2019: 866 complaints found
Toyota Highlander 2020: 292 complaints found


Toyota Tacoma 2019: 209 complaints found
Honda Accord 2019: 637 complaints found


Honda Civic 2019: 348 complaints found


Honda CR-V 2019: 1056 complaints found
Honda Pilot 2019: 849 complaints found


...

Volkswagen Tiguan 2019: 182 complaints found
Volkswagen Atlas 2019: 459 complaints found


Kia Telluride 2020: 726 complaints found
Kia Sorento 2019: 413 complaints found


Kia Forte 2019: 145 complaints found
Mazda CX-5 2019: 258 complaints found
Mazda Mazda3 2019: 72 complaints found


Acura MDX 2019: 181 complaints found
Chrysler Pacifica 2019: 332 complaints found
Infiniti QX60 2019: 35 complaints found


Buick Encore 2019: 78 complaints found
total records: 216


In [3]:
# Gemini embeddings, 768 dims to match the Pinecone index.
# Key goes in a header (not the URL) so it never ends up in logs or errors.
EMBED_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents"
HEADERS = {"x-goog-api-key": GEMINI_KEY, "Content-Type": "application/json"}

# Put the vehicle name in the embedded text so a query like "Mercedes problems"
# matches the right car, not just any complaint about "problems".
def embed_text(r):
    return f"{r['year']} {r['make']} {r['model']} - {', '.join(r['components'])}. {r['summary']}"

def embed_batch(texts):
    body = {"requests": [
        {"model": "models/gemini-embedding-001",
         "content": {"parts": [{"text": t}]},
         "outputDimensionality": 768}
        for t in texts
    ]}
    for attempt in range(5):
        r = requests.post(EMBED_URL, headers=HEADERS, json=body, timeout=120)
        if r.status_code == 429:   # free tier rate limit, wait and retry
            time.sleep(15 * (attempt + 1))
            continue
        r.raise_for_status()
        return [e["values"] for e in r.json()["embeddings"]]
    raise RuntimeError("embedding failed after retries")

vectors = []
batch = 15
for i in range(0, len(records), batch):
    chunk = records[i:i+batch]
    embs = embed_batch([embed_text(r) for r in chunk])
    for r, emb in zip(chunk, embs):
        meta = {k: r[k] for k in ("make","model","year","components","date","crash","fire","injuries","deaths","summary")}
        vectors.append({"id": r["id"], "values": emb, "metadata": meta})
    print(f"embedded {len(vectors)}/{len(records)}")
    time.sleep(12)

embedded 15/216


embedded 30/216


embedded 45/216


embedded 60/216


embedded 75/216


...

embedded 165/216


embedded 180/216


embedded 195/216


embedded 210/216


embedded 216/216


In [4]:
pc = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)
index.delete(delete_all=True)   # clear so re-running gives a clean index
index.upsert(vectors=vectors)
time.sleep(2)
print("upserted", len(vectors), "vectors into", INDEX_NAME)
print(index.describe_index_stats())

upserted 216 vectors into carverdict
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 216}},
 'total_vector_count': 216,
 'vector_type': 'dense'}


In [5]:
with open("client-complaints.json", "w") as f:
    json.dump(records, f, indent=2)
print("saved", len(records), "records to client-complaints.json")

saved 216 records to client-complaints.json
